In [1]:
import numpy as np
import igl
import pyvista as pv
pv.set_jupyter_backend('trame')

In [2]:
def to_pyvista_mesh(V, F = None):
    if F is None:
        return pv.PolyData(V)
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

In [3]:
# Utility function to generate a tet grid
# n is a 3-tuple with the number of cell in every direction
# mmin/mmax are the grid bounding box corners

def tet_grid(n, mmin, mmax):
    nx = n[0]
    ny = n[1]
    nz = n[2]

    delta = mmax-mmin

    deltax = delta[0]/(nx-1)
    deltay = delta[1]/(ny-1)
    deltaz = delta[2]/(nz-1)

    T = np.zeros(((nx-1)*(ny-1)*(nz-1)*6, 4), dtype=np.int64)
    V = np.zeros((nx*ny*nz, 3))

    mapping = -np.ones((nx, ny, nz), dtype=np.int64)


    index = 0
    for i in range(nx):
        for j in range(ny):
            for k in range(nz):
                mapping[i, j, k] = index
                V[index, :] = [i*deltax, j*deltay, k*deltaz]
                index += 1
    assert(index == V.shape[0])

    tets = np.array([
        [0,1,3,4],
        [5,2,6,7],
        [4,1,5,3],
        [4,3,7,5],
        [3,1,5,2],
        [2,3,7,5]
    ])

    index = 0
    for i in range(nx-1):
        for j in range(ny-1):
            for k in range(nz-1):
                indices = [
                    (i,   j,   k),
                    (i+1, j,   k),
                    (i+1, j+1, k),
                    (i,   j+1, k),

                    (i,   j,   k+1),
                    (i+1, j,   k+1),
                    (i+1, j+1, k+1),
                    (i,   j+1, k+1),
                ]

                for t in range(tets.shape[0]):
                    tmp = [mapping[indices[ii]] for ii in tets[t, :]]
                    T[index, :]=tmp
                    index += 1

    assert(index == T.shape[0])

    V += mmin
    return V, T

# Reading point cloud

In [4]:
pi, v = igl.read_triangle_mesh("data/cat.off")
pi /= 10
ni = igl.per_vertex_normals(pi, v)
to_pyvista_mesh(pi).plot()

Widget(value='<iframe src="http://localhost:60900/index.html?ui=P_0x27e365ffd30_0&reconnect=auto" class="pyvis…

In [5]:
def find_closest_point(point, points):
    # This works as long as 'points' stays a 2D matrix (Nx3)
    distances = np.linalg.norm(points - point, axis=1)
    return np.argmin(distances)

nPoints = pi.shape[0]
mins = pi.min(axis=0)
maxs = pi.max(axis=0)
bbox_diag = np.linalg.norm(maxs - mins)
epsilonConstant = 0.01 * bbox_diag
niNormalized = ni / np.linalg.norm(ni, axis=1, keepdims=True)

constraintsP = []
constraintsD = []
colours = []

for i in range(nPoints):
    # IMPORTANT: Use p_i (singular) so we don't destroy pi (the whole list)
    p_i = pi[i] 
    n_i = niNormalized[i]

    # 1. Surface constraint
    constraintsP.append(p_i)
    constraintsD.append(0.0)
    colours.append(0)

    # 2. Outside constraint
    epsilon = epsilonConstant
    while True:
        pPlus = p_i + epsilon * n_i
        # We check against the FULL 'pi' matrix and the current index 'i'
        # print("pi shape:", np.asarray(pi).shape)
        # print("point shape:", np.asarray(pPlus).shape)
        if find_closest_point(pPlus, pi) == i:
            constraintsP.append(pPlus)
            constraintsD.append(epsilon)
            colours.append(1) # Red
            break
        epsilon /= 2.0

    # 3. Inside constraint
    epsilon = epsilonConstant
    while True:
        pMinus = p_i - epsilon * n_i
        if find_closest_point(pMinus, pi) == i:
            constraintsP.append(pMinus)
            constraintsD.append(-epsilon)
            colours.append(-1) # Green
            break
        epsilon /= 2.0

# Wrap them up into the final arrays
P = np.array(constraintsP)
D = np.array(constraintsD)
C = np.array(colours)

# Plotting
cloud = to_pyvista_mesh(P)
cloud.point_data['side'] = C
pl = pv.Plotter()
pl.add_mesh(cloud, scalars='side', cmap=['green', 'blue', 'red'], 
            point_size=3.0, render_points_as_spheres=True)
pl.show()

Widget(value='<iframe src="http://localhost:60900/index.html?ui=P_0x27e3af9a740_1&reconnect=auto" class="pyvis…

# MLS function

In [ ]:
# Parameters
bbox_min = np.array([-1., -1., -1.])
bbox_max = np.array([1., 1., 1.])
bbox_diag = np.linalg.norm(bbox_max - bbox_min)

n = 10

In [ ]:
resolution = 30          
polyDegree = 1           
wendlandRadius = 0.2 * bbox_diag

bbox_min = pi.min(axis=0)
bbox_max = pi.max(axis=0)
bbox_diag = np.linalg.norm(bbox_max - bbox_min)

padding = 0.05 * bbox_diag
bbox_min -= padding
bbox_max += padding

x, T = tet_grid((resolution, resolution, resolution), bbox_min, bbox_max)


# --- Neighborhood query ---
def closest_points(point, points, h):
    dists = np.linalg.norm(points - point, axis=1)
    return np.argwhere(dists < h).flatten()


# --- Wendland weight ---
def wendland_weight(r):
    w = np.zeros_like(r)
    mask = r < 1.0
    rm = r[mask]
    w[mask] = (1 - rm)**4 * (4*rm + 1)
    return w


# --- Polynomial basis ---
def poly_basis(x, degree):
    if degree == 0:
        return np.array([1.0])
    elif degree == 1:
        return np.array([1.0, x[0], x[1], x[2]])
    elif degree == 2:
        return np.array([
            1.0,
            x[0], x[1], x[2],
            x[0]**2, x[1]**2, x[2]**2,
            x[0]*x[1], x[0]*x[2], x[1]*x[2]
        ])
    else:
        raise ValueError("Unsupported polynomial degree")


# --- MLS evaluation at one point ---
def evaluate_mls(xi, P, D, h, polyDegree):
    idx = closest_points(xi, P, h)

    n_coeffs = {0: 1, 1: 4, 2: 10}[polyDegree]

    # Not enough constraints → outside
    if len(idx) < 2 * n_coeffs:
        return 1.0

    Pi = P[idx]
    Di = D[idx]

    A = np.array([poly_basis(p, polyDegree) for p in Pi])

    r = np.linalg.norm(Pi - xi, axis=1) / h
    w = wendland_weight(r)
    W = np.diag(w)

    ATA = A.T @ W @ A
    ATb = A.T @ W @ Di

    try:
        c = np.linalg.solve(ATA, ATb)
    except np.linalg.LinAlgError:
        return 1.0

    return poly_basis(xi, polyDegree) @ c


# --- Evaluate field on grid ---
fx = np.zeros(x.shape[0])

for i in range(x.shape[0]):
    fx[i] = evaluate_mls(
        x[i],
        P,
        D,
        wendlandRadius,
        polyDegree
    )


# --- Visualize inside / outside ---
ind = np.zeros_like(fx)
ind[fx >= 0] = 1
ind[fx < 0] = -1

to_pyvista_mesh(x).plot(scalars=ind)

NameError: name 'polyDegree' is not defined

In [ ]:
# Generate grid n x n x n

x, T = tet_grid((n, n, n), bbox_min - 0.05 * bbox_diag, bbox_max + 0.05 * bbox_diag)

#Compute implicit sphere function
center = np.array([0., 0., 0.])
radius = 1
fx = np.linalg.norm(x-center, axis=1) - radius

In [ ]:
# Treshold fx to visualize inside outside

ind = np.zeros_like(fx)
ind[fx >= 0] = 1
ind[fx < 0] = -1

to_pyvista_mesh(x).plot(scalars=ind)

Widget(value='<iframe src="http://localhost:60900/index.html?ui=P_0x27e3af9b8b0_2&reconnect=auto" class="pyvis…

# Marching to extract surface

In [ ]:
# Marcing tet to extract surface

sv, sf, _, _ = igl.marching_tets(x, T, fx, 0)

to_pyvista_mesh(sv, sf).plot(show_edges=True)

AttributeError: module 'igl' has no attribute 'marching_tets'